# 🐍 TUTORIAL INTERACTIVO - MODELO ECHEVESTE 2020

## 📋 Guía Completa para Ejecutar Ejemplos

Este notebook te permitirá ejecutar todos los ejemplos tanto del **código original** como de **nuestra implementación**.

---

## 🔧 CONFIGURACIÓN INICIAL

In [ ]:
import sys
import numpy as np
import os
import matplotlib.pyplot as plt

# Configurar paths
sys.path.insert(0, '/home/molina/FAMAF/5to-Famaf/TESIS/scikit-neuromsi')
sys.path.insert(0, '/home/molina/FAMAF/5to-Famaf/TESIS/ssn_inference_numerical_experiments/GSM')

print("✅ Imports y paths configurados correctamente")

---
## 📊 EJEMPLO 1: CÓDIGO ORIGINAL GSM

Ejecutamos el código original de Echeveste et al. (2020) para generar observaciones usando el modelo GSM.

In [ ]:
# Cambiar al directorio del código original
original_dir = os.getcwd()
os.chdir('/home/molina/FAMAF/5to-Famaf/TESIS/ssn_inference_numerical_experiments/GSM')

try:
    # Importar GSM original
    import GSM
    
    # Cargar filtros originales
    A = np.load("filters.npy")
    print(f"📊 Filtros cargados: {A.shape}")
    
    # Parámetros del ejemplo
    z = 0.32  # Contraste
    s_x = 10.0  # Nivel de ruido
    
    # Generar estímulo de orientación
    D_y = A.shape[1]  # 50 orientaciones
    y = np.zeros(D_y)
    y[0] = 1.0  # Orientación horizontal
    
    # Generar observación usando GSM original
    x = GSM.get_x(y, z, A, s_x)
    
    # Mostrar resultados
    print(f"\n🔍 RESULTADOS CÓDIGO ORIGINAL:")
    print(f"   Forma de x: {x.shape}")
    print(f"   Media: {np.mean(x):.4f}")
    print(f"   Desviación: {np.std(x):.4f}")
    print(f"   Rango: [{np.min(x):.4f}, {np.max(x):.4f}]")
    
    # Visualizar
    plt.figure(figsize=(12, 4))
    
    plt.subplot(1, 3, 1)
    plt.plot(x)
    plt.title('Observación x (Original)')
    plt.xlabel('Píxel')
    plt.ylabel('Intensidad')
    
    plt.subplot(1, 3, 2)
    plt.hist(x, bins=30, alpha=0.7)
    plt.title('Distribución de x')
    plt.xlabel('Intensidad')
    plt.ylabel('Frecuencia')
    
    plt.subplot(1, 3, 3)
    plt.imshow(x.reshape(16, 16), cmap='viridis')
    plt.title('x como imagen 16x16')
    plt.colorbar()
    
    plt.tight_layout()
    plt.show()

except Exception as e:
    print(f"❌ Error: {e}")

finally:
    os.chdir(original_dir)

---
## 🚀 EJEMPLO 2: NUESTRA IMPLEMENTACIÓN SSN

Ejecutamos nuestra implementación del modelo Echeveste2020 con red SSN completa.

In [ ]:
# Importar nuestra implementación
from skneuromsi.neural import Echeveste2020

# Crear instancia del modelo
ssn = Echeveste2020(N_E=50, N_I=50, seed=42)

# Cargar parámetros pre-entrenados
ssn.load_parameters('/home/molina/FAMAF/5to-Famaf/TESIS/scikit-neuromsi/skneuromsi/data/echeveste2020/')

# Ejecutar simulación
response = ssn.run(
    stimulus_contrast=0.32,
    stimulus_orientation=0.0,
    simulation_time=1000.0,
    noise_level=0.1
)

# Extraer actividad
df = response.get_modes()
activity_exc = df['excitatory'].values.reshape(5000, 50)
activity_inh = df['inhibitory'].values.reshape(5000, 50)

# Mostrar resultados
print(f"🔍 RESULTADOS NUESTRA IMPLEMENTACIÓN:")
print(f"   Forma de actividad: {activity_exc.shape}")
print(f"   Media excitatory: {np.mean(activity_exc):.4f}")
print(f"   Desviación excitatory: {np.std(activity_exc):.4f}")
print(f"   Rango excitatory: [{np.min(activity_exc):.4f}, {np.max(activity_exc):.4f}]")

# Actividad final (steady-state)
final_activity = activity_exc[-1000:]
print(f"   Media final: {np.mean(final_activity):.4f}")

# Visualizar
plt.figure(figsize=(15, 5))

plt.subplot(1, 4, 1)
plt.plot(activity_exc[:1000, :5])
plt.title('Actividad Excitatory (primeras 5 neuronas)')
plt.xlabel('Tiempo')
plt.ylabel('Actividad')

plt.subplot(1, 4, 2)
plt.plot(activity_inh[:1000, :5])
plt.title('Actividad Inhibitory (primeras 5 neuronas)')
plt.xlabel('Tiempo')
plt.ylabel('Actividad')

plt.subplot(1, 4, 3)
plt.hist(activity_exc.flatten(), bins=50, alpha=0.7)
plt.title('Distribución Actividad Excitatory')
plt.xlabel('Actividad')
plt.ylabel('Frecuencia')

plt.subplot(1, 4, 4)
plt.imshow(activity_exc[-100:].T, aspect='auto', cmap='viridis')
plt.title('Actividad Final (últimos 100 pasos)')
plt.xlabel('Tiempo')
plt.ylabel('Neurona')
plt.colorbar()

plt.tight_layout()
plt.show()

---
## 🧠 EJEMPLO 3: INFERENCIA CAUSAL

Usamos la actividad de red para inferir las causas presentes en la escena.

In [ ]:
# Calcular estado de red promedio
mean_exc = np.mean(activity_exc, axis=0)  # Promedio temporal
mean_inh = np.mean(activity_inh, axis=0)

# Combinar E + I para red completa
network_state = np.concatenate([mean_exc, mean_inh])

print(f"📊 Estado de red para inferencia:")
print(f"   Forma del estado: {network_state.shape}")
print(f"   Media: {np.mean(network_state):.4f}")

# Inferencia causal
causes = ssn.calculate_causes(
    network_activity=network_state,
    confidence_threshold=0.95
)

# Mostrar resultados de inferencia
print(f"\n🧠 RESULTADOS INFERENCIA CAUSAL:")
print(f"   Número de causas: {causes['num_causes']}")
print(f"   Posiciones: {causes['cause_positions']}")
print(f"   Contrastes: {causes['cause_contrasts']}")
print(f"   Confianza: {causes['confidence']:.4f}")

# Visualizar distribución posterior
posterior = causes['posterior_distribution']
orientations = np.linspace(0, 180, len(posterior))

plt.figure(figsize=(12, 4))

plt.subplot(1, 3, 1)
plt.bar(range(len(mean_exc)), mean_exc)
plt.title('Estado Promedio Excitatory')
plt.xlabel('Neurona')
plt.ylabel('Actividad Media')

plt.subplot(1, 3, 2)
plt.bar(range(len(mean_inh)), mean_inh)
plt.title('Estado Promedio Inhibitory')
plt.xlabel('Neurona')
plt.ylabel('Actividad Media')

plt.subplot(1, 3, 3)
plt.plot(orientations, posterior)
plt.title('Distribución Posterior de Causas')
plt.xlabel('Orientación (grados)')
plt.ylabel('Probabilidad Posterior')

# Marcar causas detectadas
for pos, contrast in zip(causes['cause_positions'], causes['cause_contrasts']):
    orientation = pos * 180
    plt.axvline(orientation, color='red', linestyle='--', alpha=0.7)
    plt.text(orientation, max(posterior)*0.8, f'{contrast:.2f}', 
             rotation=90, ha='center', va='bottom')

plt.tight_layout()
plt.show()

---
## 📈 EJEMPLO 4: COMPARACIÓN DE ESCALAS

Comparamos las escalas numéricas entre ambas implementaciones.

In [ ]:
# Probar diferentes contrastes para comparación
contrastes = [0.1, 0.32, 0.8]
resultados_original = []
resultados_nuestro = []

print("📈 COMPARACIÓN DE ESCALAS:")
print("Contraste | Original | Nuestro | Ratio")
print("-" * 40)

for z in contrastes:
    # Original
    os.chdir('/home/molina/FAMAF/5to-Famaf/TESIS/ssn_inference_numerical_experiments/GSM')
    try:
        y = np.zeros(50)
        y[0] = 1.0
        x_orig = GSM.get_x(y, z, A, 10.0)
        orig_mean = np.mean(x_orig)
    except:
        orig_mean = np.nan

    # Nuestro
    os.chdir('/home/molina/FAMAF/5to-Famaf/TESIS')
    ssn_temp = Echeveste2020(N_E=50, N_I=50, seed=42)
    ssn_temp.load_parameters('/home/molina/FAMAF/5to-Famaf/TESIS/scikit-neuromsi/skneuromsi/data/echeveste2020/')

    resp = ssn_temp.run(stimulus_contrast=z, stimulus_orientation=0.0, simulation_time=1000.0)
    act = resp.get_modes()['excitatory'].values.reshape(5000, 50)
    our_mean = np.mean(act[-1000:])  # Steady state

    ratio = orig_mean / our_mean if our_mean != 0 and not np.isnan(orig_mean) else float('inf')

    print(f"{z:8.2f} | {orig_mean:8.3f} | {our_mean:7.3f} | {ratio:5.1f}x")

    resultados_original.append(orig_mean)
    resultados_nuestro.append(our_mean)

os.chdir(original_dir)

# Visualizar comparación
plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.plot(contrastes, resultados_original, 'o-', label='Original', linewidth=2)
plt.plot(contrastes, resultados_nuestro, 's-', label='Nuestro', linewidth=2)
plt.xlabel('Contraste')
plt.ylabel('Actividad Media')
plt.title('Comparación de Escalas')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
ratios = [o/n if n != 0 and not np.isnan(o) else 0 for o, n in zip(resultados_original, resultados_nuestro)]
plt.bar(range(len(contrastes)), ratios, alpha=0.7)
plt.xlabel('Índice de Contraste')
plt.ylabel('Ratio (Original/Nuestro)')
plt.title('Ratios de Escala')
plt.xticks(range(len(contrastes)), [f'{c:.1f}' for c in contrastes])
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 🔬 EJEMPLO 5: TEST DE VARIABILIDAD

Analizamos la variabilidad estocástica en múltiples ejecuciones.

In [ ]:
# Test de variabilidad con múltiples ejecuciones
print("🔬 TEST DE VARIABILIDAD (5 ejecuciones):")
print("Run | Media | Std | Causas")
print("-" * 30)

variabilidad_resultados = []
actividades_todas = []

for i in range(5):
    # Nueva instancia con diferente seed
    ssn_var = Echeveste2020(N_E=50, N_I=50, seed=42+i)
    ssn_var.load_parameters('/home/molina/FAMAF/5to-Famaf/TESIS/scikit-neuromsi/skneuromsi/data/echeveste2020/')

    # Simulación
    resp = ssn_var.run(
        stimulus_contrast=0.32,
        stimulus_orientation=0.0,
        simulation_time=1000.0,
        noise_level=0.1
    )

    # Análisis
    df_var = resp.get_modes()
    act_var = df_var['excitatory'].values.reshape(5000, 50)
    actividades_todas.append(act_var)

    # Inferencia causal
    mean_e = np.mean(act_var, axis=0)
    mean_i = np.mean(df_var['inhibitory'].values.reshape(5000, 50), axis=0)
    net_state = np.concatenate([mean_e, mean_i])

    causes_var = ssn_var.calculate_causes(network_activity=net_state)

    # Guardar resultados
    result = {
        'run': i+1,
        'mean': np.mean(act_var),
        'std': np.std(act_var),
        'n_causes': causes_var['num_causes']
    }

    variabilidad_resultados.append(result)

    print(f"{i+1:3d} | {result['mean']:.3f} | {result['std']:.2f} | {result['n_causes']:6d}")

# Análisis de variabilidad
medias = [r['mean'] for r in variabilidad_resultados]
stds = [r['std'] for r in variabilidad_resultados]
cv = np.std(medias) / np.mean(medias) * 100

print(f"\n📊 ANÁLISIS DE VARIABILIDAD:")
print(f"   Coeficiente de variación: {cv:.2f}%")
print(f"   Rango de medias: [{np.min(medias):.3f}, {np.max(medias):.3f}]")
print(f"   Rango de stds: [{np.min(stds):.3f}, {np.max(stds):.3f}]")

# Visualizar variabilidad
plt.figure(figsize=(12, 8))

# Actividad de todas las ejecuciones
plt.subplot(2, 3, 1)
for i, act in enumerate(actividades_todas):
    plt.plot(act[:500, 0], alpha=0.7, label=f'Run {i+1}')
plt.title('Primera Neurona - Todas las Ejecuciones')
plt.xlabel('Tiempo')
plt.ylabel('Actividad')
plt.legend()

# Distribución de medias
plt.subplot(2, 3, 2)
plt.hist(medias, bins=10, alpha=0.7)
plt.title('Distribución de Medias')
plt.xlabel('Media de Actividad')
plt.ylabel('Frecuencia')

# Distribución de stds
plt.subplot(2, 3, 3)
plt.hist(stds, bins=10, alpha=0.7)
plt.title('Distribución de Desviaciones')
plt.xlabel('Desviación Estándar')
plt.ylabel('Frecuencia')

# Box plot de medias
plt.subplot(2, 3, 4)
plt.boxplot(medias)
plt.title('Box Plot - Medias')
plt.ylabel('Media de Actividad')

# Actividad final promedio
plt.subplot(2, 3, 5)
actividad_final_promedio = np.mean([act[-100:] for act in actividades_todas], axis=0)
plt.imshow(actividad_final_promedio.T, aspect='auto', cmap='viridis')
plt.title('Actividad Final Promedio')
plt.xlabel('Tiempo')
plt.ylabel('Neurona')
plt.colorbar()

# CV por neurona
plt.subplot(2, 3, 6)
cv_por_neurona = []
for j in range(50):
    medias_neurona = [np.mean(act[:, j]) for act in actividades_todas]
    cv_neurona = np.std(medias_neurona) / np.mean(medias_neurona) * 100
    cv_por_neurona.append(cv_neurona)

plt.bar(range(50), cv_por_neurona)
plt.title('CV por Neurona')
plt.xlabel('Neurona')
plt.ylabel('CV (%)')

plt.tight_layout()
plt.show()

---
## 🎯 EJEMPLO 6: COMPARACIÓN DIRECTA COMPLETA

Función para hacer comparación completa entre implementaciones.

In [ ]:
def comparar_implementaciones_completa(contrast=0.32, noise=0.1):
    """
    Función para comparación completa entre implementaciones.
    """
    print(f"🎯 COMPARACIÓN DIRECTA COMPLETA")
    print(f"Parámetros: contrast={contrast}, noise={noise}")
    print("=" * 60)

    resultados = {'original': {}, 'nuestro': {}}
    
    # Original
    os.chdir('/home/molina/FAMAF/5to-Famaf/TESIS/ssn_inference_numerical_experiments/GSM')
    try:
        y = np.zeros(50)
        y[0] = 1.0
        x_orig = GSM.get_x(y, contrast, A, 10.0)

        resultados['original'] = {
            'data': x_orig,
            'mean': np.mean(x_orig),
            'std': np.std(x_orig),
            'min': np.min(x_orig),
            'max': np.max(x_orig),
            'shape': x_orig.shape,
            'success': True
        }
    except Exception as e:
        print(f"❌ Error en original: {e}")
        resultados['original']['success'] = False

    # Nuestro
    os.chdir('/home/molina/FAMAF/5to-Famaf/TESIS')
    try:
        ssn_comp = Echeveste2020(N_E=50, N_I=50, seed=42)
        ssn_comp.load_parameters('/home/molina/FAMAF/5to-Famaf/TESIS/scikit-neuromsi/skneuromsi/data/echeveste2020/')

        resp = ssn_comp.run(
            stimulus_contrast=contrast,
            stimulus_orientation=0.0,
            simulation_time=1000.0,
            noise_level=noise
        )

        act = resp.get_modes()['excitatory'].values.reshape(5000, 50)

        resultados['nuestro'] = {
            'data': act,
            'mean': np.mean(act),
            'std': np.std(act),
            'min': np.min(act),
            'max': np.max(act),
            'shape': act.shape,
            'success': True
        }
    except Exception as e:
        print(f"❌ Error en nuestro: {e}")
        resultados['nuestro']['success'] = False

    os.chdir(original_dir)
    
    # Mostrar comparación
    if resultados['original']['success'] and resultados['nuestro']['success']:
        orig = resultados['original']
        nues = resultados['nuestro']
        
        print("\nMétrica      | Original | Nuestro | Ratio")
        print("-" * 50)
        print(f"Forma        | {str(orig['shape']):8} | {str(nues['shape']):7} | -")
        print(f"Media        | {orig['mean']:8.3f} | {nues['mean']:7.3f} | {orig['mean']/nues['mean']:5.1f}x")
        print(f"Desviación   | {orig['std']:8.3f} | {nues['std']:7.3f} | {orig['std']/nues['std']:5.1f}x")
        print(f"Mínimo       | {orig['min']:8.3f} | {nues['min']:7.3f} | -")
        print(f"Máximo       | {orig['max']:8.3f} | {nues['max']:7.3f} | {orig['max']/nues['max']:5.1f}x")
        
        # Visualización comparativa
        plt.figure(figsize=(15, 10))
        
        # Datos originales
        plt.subplot(3, 4, 1)
        plt.plot(orig['data'])
        plt.title('Original: Señal Completa')
        plt.xlabel('Píxel')
        plt.ylabel('Intensidad')
        
        plt.subplot(3, 4, 2)
        plt.hist(orig['data'], bins=30, alpha=0.7)
        plt.title('Original: Distribución')
        plt.xlabel('Intensidad')
        plt.ylabel('Frecuencia')
        
        plt.subplot(3, 4, 3)
        if len(orig['data']) == 256:
            plt.imshow(orig['data'].reshape(16, 16), cmap='viridis')
            plt.title('Original: Como Imagen 16x16')
            plt.colorbar()
        else:
            plt.text(0.5, 0.5, 'No es 256D', ha='center', va='center', transform=plt.gca().transAxes)
            plt.title('Original: Forma inesperada')
        
        plt.subplot(3, 4, 4)
        plt.text(0.1, 0.8, f"Media: {orig['mean']:.3f}", transform=plt.gca().transAxes)
        plt.text(0.1, 0.6, f"Std: {orig['std']:.3f}", transform=plt.gca().transAxes)
        plt.text(0.1, 0.4, f"Min: {orig['min']:.3f}", transform=plt.gca().transAxes)
        plt.text(0.1, 0.2, f"Max: {orig['max']:.3f}", transform=plt.gca().transAxes)
        plt.title('Original: Estadísticas')
        plt.axis('off')
        
        # Nuestros datos
        plt.subplot(3, 4, 5)
        plt.plot(nues['data'][:1000, 0])  # Primera neurona, primeros 1000 pasos
        plt.title('Nuestro: Primera Neurona')
        plt.xlabel('Tiempo')
        plt.ylabel('Actividad')
        
        plt.subplot(3, 4, 6)
        plt.hist(nues['data'].flatten(), bins=30, alpha=0.7)
        plt.title('Nuestro: Distribución')
        plt.xlabel('Actividad')
        plt.ylabel('Frecuencia')
        
        plt.subplot(3, 4, 7)
        plt.imshow(nues['data'][-100:].T, aspect='auto', cmap='viridis')
        plt.title('Nuestro: Actividad Final')
        plt.xlabel('Tiempo')
        plt.ylabel('Neurona')
        plt.colorbar()
        
        plt.subplot(3, 4, 8)
        plt.text(0.1, 0.8, f"Media: {nues['mean']:.3f}", transform=plt.gca().transAxes)
        plt.text(0.1, 0.6, f"Std: {nues['std']:.3f}", transform=plt.gca().transAxes)
        plt.text(0.1, 0.4, f"Min: {nues['min']:.3f}", transform=plt.gca().transAxes)
        plt.text(0.1, 0.2, f"Max: {nues['max']:.3f}", transform=plt.gca().transAxes)
        plt.title('Nuestro: Estadísticas')
        plt.axis('off')
        
        # Comparaciones
        plt.subplot(3, 4, 9)
        medias = [orig['mean'], nues['mean']]
        plt.bar(['Original', 'Nuestro'], medias, alpha=0.7)
        plt.title('Comparación: Medias')
        plt.ylabel('Media')
        
        plt.subplot(3, 4, 10)
        stds = [orig['std'], nues['std']]
        plt.bar(['Original', 'Nuestro'], stds, alpha=0.7)
        plt.title('Comparación: Desviaciones')
        plt.ylabel('Desviación Estándar')
        
        plt.subplot(3, 4, 11)
        rangos = [orig['max']-orig['min'], nues['max']-nues['min']]
        plt.bar(['Original', 'Nuestro'], rangos, alpha=0.7)
        plt.title('Comparación: Rangos')
        plt.ylabel('Rango')
        
        plt.subplot(3, 4, 12)
        ratios = [orig['mean']/nues['mean'], orig['std']/nues['std']]
        plt.bar(['Media', 'Std'], ratios, alpha=0.7)
        plt.title('Ratios Orig/Nuestro')
        plt.ylabel('Ratio')
        plt.yscale('log')
        
        plt.tight_layout()
        plt.show()
    
    return resultados

# Ejecutar comparación completa
resultados_completos = comparar_implementaciones_completa(0.32, 0.1)

---
## 🛠️ UTILIDADES Y COMANDOS ÚTILES

In [ ]:
# Función para limpiar memoria y ver variables
def mostrar_estado_memoria():
    """Mostrar estado actual de variables en memoria."""
    import gc
    
    print("🧠 ESTADO DE MEMORIA:")
    print(f"   Variables globales: {len(globals())}")
    
    # Variables grandes
    variables_grandes = []
    for name, obj in globals().items():
        if hasattr(obj, '__len__') and len(obj) > 1000:
            variables_grandes.append((name, type(obj).__name__, len(obj)))
    
    if variables_grandes:
        print("   Variables grandes:")
        for name, tipo, size in variables_grandes:
            print(f"     {name}: {tipo} (size: {size})")
    
    # Limpieza
    gc.collect()
    print("   ✅ Garbage collection ejecutado")

# Función para reiniciar todo
def reiniciar_ambiente():
    """Reiniciar el ambiente de trabajo."""
    # Limpiar variables (excepto las básicas)
    vars_a_mantener = ['sys', 'np', 'os', 'plt', 'mostrar_estado_memoria', 'reiniciar_ambiente']
    
    for var in list(globals().keys()):
        if var not in vars_a_mantener and not var.startswith('_'):
            del globals()[var]
    
    print("🔄 Ambiente reiniciado")
    print("   Ejecuta la celda de configuración inicial para recomenzar")

# Función para diagnóstico rápido
def diagnostico_rapido():
    """Hacer diagnóstico rápido del estado del sistema."""
    print("🔍 DIAGNÓSTICO RÁPIDO:")
    
    # Verificar paths
    paths_necesarios = [
        '/home/molina/FAMAF/5to-Famaf/TESIS/scikit-neuromsi',
        '/home/molina/FAMAF/5to-Famaf/TESIS/ssn_inference_numerical_experiments/GSM'
    ]
    
    for path in paths_necesarios:
        exists = os.path.exists(path)
        print(f"   {path}: {'✅' if exists else '❌'}")
    
    # Verificar imports
    try:
        from skneuromsi.neural import Echeveste2020
        print("   Echeveste2020 import: ✅")
    except:
        print("   Echeveste2020 import: ❌")
    
    # Verificar GSM original
    try:
        os.chdir('/home/molina/FAMAF/5to-Famaf/TESIS/ssn_inference_numerical_experiments/GSM')
        import GSM
        A = np.load("filters.npy")
        print(f"   GSM original: ✅ (filtros: {A.shape})")
        os.chdir('/home/molina/FAMAF/5to-Famaf/TESIS')
    except:
        print("   GSM original: ❌")

# Ejecutar diagnóstico
diagnostico_rapido()
mostrar_estado_memoria()

---
## 📝 RESUMEN DEL TUTORIAL

### 🎯 Ejemplos Completados:

1. **Código Original GSM**: Generación de observaciones visuales
2. **Nuestra Implementación SSN**: Simulación de red neuronal completa
3. **Inferencia Causal**: Detección de causas desde actividad de red
4. **Comparación de Escalas**: Análisis de diferencias numéricas
5. **Test de Variabilidad**: Robustez estadística
6. **Comparación Directa**: Análisis completo lado a lado

### 🔧 Herramientas Disponibles:

- `mostrar_estado_memoria()`: Ver variables en memoria
- `reiniciar_ambiente()`: Limpiar y reiniciar
- `diagnostico_rapido()`: Verificar estado del sistema
- `comparar_implementaciones_completa()`: Comparación detallada

### 📊 Resultados Típicos:

- **Original**: x ∈ [-54, +53], media ~0.12, std ~10.37
- **Nuestro**: activity ∈ [0, ~10], media ~0.40, std ~0.82
- **Inferencia**: ~6 causas detectadas consistentemente
- **Variabilidad**: CV ~3% (muy estable)

### 🎉 ¡Tutorial Completo!

Ahora tienes todas las herramientas para:
- Ejecutar ambas implementaciones
- Comparar resultados
- Analizar diferencias
- Hacer experimentos personalizados

**💡 Sugerencia**: Experimenta con diferentes parámetros en cada ejemplo para explorar el comportamiento del modelo.